# Black Swan Robustness Analysis

Reads `results/blackswan/*.json` and plots recovery curves plus recovery-step summaries.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULT_DIR = PROJECT_ROOT / 'results' / 'blackswan'

results = []
for fpath in sorted(RESULT_DIR.glob('*.json')):
    with open(fpath, encoding='utf-8') as handle:
        results.append(json.load(handle))

pd.DataFrame(results).drop(columns=['rolling_mse'], errors='ignore') if results else pd.DataFrame()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
shift_types = ['level', 'variance', 'trend', 'spike']
colors = {'no_tta': 'gray', 'original': '#378ADD', 'cosa_plus': '#E85D24'}

for ax, shift_type in zip(axes.flat, shift_types):
    for variant in ['no_tta', 'original', 'cosa_plus']:
        match = [
            r for r in results
            if r['shift_type'] == shift_type
            and r['variant'] == variant
            and float(r['magnitude']) == 3.0
        ]
        if not match:
            continue
        record = match[0]
        rolling = np.array(record['rolling_mse'])
        ax.plot(rolling, label=variant, color=colors[variant], linewidth=1.5)
        ax.axvline(record['t_shift'], color='red', linestyle='--', alpha=0.5, linewidth=1)
    ax.set_title(shift_type)
    ax.set_xlabel('Test step')
    ax.set_ylabel('Rolling MSE (w=20)')
    ax.legend(fontsize=9)

fig.suptitle('COSA vs COSA+ under abrupt distribution shifts (ETTh1, DLinear, L=96, mag=3 sigma)')
fig.tight_layout()
RESULT_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(RESULT_DIR / 'recovery_curves.png', dpi=150)
plt.show()

In [ ]:
rows = []
for record in results:
    if float(record['magnitude']) == 3.0:
        rows.append({
            'shift_type': record['shift_type'],
            'variant': record['variant'],
            'mse_after_50': record.get('mse_after_50', record.get('mse_after')),
            'mse_after_100': record.get('mse_after_100'),
            'half_recovery_steps': record.get('half_recovery_steps', record.get('recovery_steps')),
        })

df = pd.DataFrame(rows)
if df.empty:
    print('No magnitude=3.0 black swan results found yet.')
else:
    mse_pivot = df.pivot_table(index='shift_type', columns='variant', values='mse_after_50', aggfunc='mean')
    recovery_pivot = df.pivot_table(index='shift_type', columns='variant', values='half_recovery_steps', aggfunc='mean')
    print('MSE after 50 steps')
    print(mse_pivot.to_string())
    print('\nHalf recovery steps')
    print(recovery_pivot.to_string())
    mse_pivot = mse_pivot.reindex(columns=[c for c in ['no_tta', 'original', 'cosa_plus'] if c in mse_pivot.columns])
    mse_pivot.plot(kind='bar', figsize=(8, 4), color=['gray', '#378ADD', '#E85D24'][:len(mse_pivot.columns)])
    plt.ylabel('MSE after 50 steps (lower = better)')
    plt.title('Post-shift fixed-window error')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(RESULT_DIR / 'mse_after_50.png', dpi=150)
    plt.show()